In [ ]:
import subprocess, sys
for pkg in ["arch", "scikit-learn", "statsmodels", "openpyxl", "ruptures"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    print(f"  ✓ {pkg}")
print("All packages ready.")


In [ ]:
# ===========================================================
# EMIF Project — Has the structure of risk changed post-COVID?
# ===========================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller
from arch import arch_model
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Global style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# ── COVID breakpoint: WHO pandemic declaration ─────────────────
COVID_BREAK = pd.Timestamp('2020-03-11')
print(f'COVID breakpoint: {COVID_BREAK.date()} (WHO Pandemic Declaration)')


## Step 1 — Data Loading & Cleaning

In [ ]:
# ── Load ──────────────────────────────────────────────────────
df = pd.read_excel('Data.xlsx')
df.columns = [col.strip() for col in df.columns]
df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])

asset_cols = df.columns.drop('Date')
for col in asset_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('Date').set_index('Date')

# ── Missing values ─────────────────────────────────────────────
# Backward fill first (handles assets starting later in the sample)
# then forward fill (handles weekends / holidays)
df = df.bfill().ffill()

print(f'Dataset shape: {df.shape}')
print(f'Period: {df.index[0].date()} → {df.index[-1].date()}')
print(df.head(3))


## Step 2 — Returns Calculation

- **Price / index / FX / commodity series** → log-returns × 100 (in %)
- **Yield series** → first differences × 100 (basis-point changes)

Scaling by 100 aids GARCH numerical convergence.

In [ ]:
# Yield columns: first differences are more meaningful than log-returns
YIELD_COLS = ['US T 10-year Yield', 'German Gov 10-year yield']
PRICE_COLS  = [c for c in asset_cols if c not in YIELD_COLS]

returns = pd.DataFrame(index=df.index)
for col in PRICE_COLS:
    returns[col] = np.log(df[col] / df[col].shift(1)) * 100
for col in YIELD_COLS:
    returns[col] = df[col].diff() * 100          # basis-point changes

returns = returns.dropna()

pre  = returns[returns.index <  COVID_BREAK]
post = returns[returns.index >= COVID_BREAK]

print(f'Returns dataset: {returns.shape}')
print(f'Pre-COVID  observations: {len(pre):,}  ({pre.index[0].date()} → {pre.index[-1].date()})')
print(f'Post-COVID observations: {len(post):,}  ({post.index[0].date()} → {post.index[-1].date()})')


## Step 3 — Stationarity Tests (ADF)

In [ ]:
print('ADF TEST — Null: unit root (non-stationary)')
print(f'{"Asset":<30} {"ADF Stat":>10} {"p-value":>10} {"Stationary?":>12}')
print('-' * 65)
for col in returns.columns:
    stat, pval, *_ = adfuller(returns[col])
    flag = 'YES' if pval < 0.05 else 'NO  ← WARNING'
    print(f'{col:<30} {stat:>10.4f} {pval:>10.4e} {flag:>12}')


## Step 5 — Descriptive Statistics: Pre vs Post COVID

In [ ]:
def describe_period(df_period, label):
    d = pd.DataFrame({
        'Mean (%)':   df_period.mean(),
        'Std (%)':    df_period.std(),
        'Skewness':   df_period.skew(),
        'Kurtosis':   df_period.kurt(),
        'Min (%)':    df_period.min(),
        'Max (%)':    df_period.max(),
    }).round(4)
    print(f'\n{'─'*65}\n  {label}\n{'─'*65}')
    print(d.to_string())
    return d

pre_stats  = describe_period(pre,  'Pre-COVID  Descriptive Statistics')
post_stats = describe_period(post, 'Post-COVID Descriptive Statistics')

# ── Annualised volatility comparison ──────────────────────────
ann_vol_pre  = pre.std()  * np.sqrt(252)
ann_vol_post = post.std() * np.sqrt(252)
vol_change   = (ann_vol_post - ann_vol_pre) / ann_vol_pre * 100

vol_df = pd.DataFrame({
    'Ann. Vol Pre (%)':    ann_vol_pre.round(2),
    'Ann. Vol Post (%)':   ann_vol_post.round(2),
    'Change (%)':          vol_change.round(2),
}).sort_values('Change (%)', ascending=False)

print('\n── Annualised Volatility: Pre vs Post COVID ─────────────────')
print(vol_df.to_string())


## Step 6 — GARCH(1,1) Conditional Volatility

For each key asset we fit a **GARCH(1,1)** model and extract conditional volatility.
This captures *time-varying* risk levels and reveals how fast volatility clusters respond to shocks.

In [ ]:
KEY_ASSETS = ['S&P500', 'Eurostoxx 50', 'Hang Seng', 'US HY Bonds',
              'Gold', 'Oil futures', 'EURUSD', 'US T 10-year Yield']

garch_vol    = pd.DataFrame(index=returns.index)
garch_params = {}

for asset in KEY_ASSETS:
    model = arch_model(returns[asset], vol='Garch', p=1, q=1,
                       dist='normal', rescale=False)
    res = model.fit(disp='off')
    garch_vol[asset] = res.conditional_volatility
    garch_params[asset] = res.params
    print(f'{asset:<25}  omega={res.params["omega"]:.5f}  '
          f'alpha={res.params["alpha[1]"]:.4f}  '
          f'beta={res.params["beta[1]"]:.4f}  '
          f'persistence={res.params["alpha[1]"]+res.params["beta[1]"]:.4f}')

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(15, 14), sharex=True)
axes = axes.flatten()

for i, asset in enumerate(KEY_ASSETS):
    axes[i].plot(garch_vol.index, garch_vol[asset],
                 color='steelblue', lw=0.7, label='Cond. Vol')
    axes[i].axvline(COVID_BREAK, color='red', linestyle='--',
                    lw=1.5, label='COVID Break')
    # shade post-COVID
    axes[i].axvspan(COVID_BREAK, garch_vol.index[-1],
                    alpha=0.06, color='red')
    axes[i].set_title(asset, fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Vol (%)', fontsize=8)
    if i == 0:
        axes[i].legend(fontsize=7)

plt.suptitle('GARCH(1,1) Conditional Volatility — All Key Assets',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('garch_volatility.png', bbox_inches='tight', dpi=120)
plt.show()


## Step 7 — Tail Risk Analysis: VaR, CVaR, Maximum Drawdown

We compare **downside risk** pre vs post COVID using:
- **Historical VaR (95%)** — the 5th percentile of the daily return distribution
- **CVaR / Expected Shortfall (95%)** — average loss beyond VaR
- **Maximum Drawdown** — largest cumulative peak-to-trough loss

In [ ]:
def historical_var(series, conf=0.95):
    return np.percentile(series, (1 - conf) * 100)

def historical_cvar(series, conf=0.95):
    var = historical_var(series, conf)
    return series[series <= var].mean()

def max_drawdown(series):
    cum = series.cumsum()
    return (cum - cum.cummax()).min()

results = []
for col in returns.columns:
    results.append({
        'Asset':       col,
        'VaR Pre':     round(historical_var(pre[col]),  3),
        'VaR Post':    round(historical_var(post[col]), 3),
        'CVaR Pre':    round(historical_cvar(pre[col]), 3),
        'CVaR Post':   round(historical_cvar(post[col]),3),
        'MDD Pre':     round(max_drawdown(pre[col]),    3),
        'MDD Post':    round(max_drawdown(post[col]),   3),
    })

tail_df = pd.DataFrame(results).set_index('Asset')
tail_df['CVaR Δ'] = (tail_df['CVaR Post'] - tail_df['CVaR Pre']).round(3)
print('TAIL RISK — Pre vs Post COVID (95% confidence level)')
print(tail_df.to_string())

# ── Bar chart: CVaR comparison ─────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(tail_df))
w = 0.35
ax.bar(x - w/2, tail_df['CVaR Pre'],  w, label='Pre-COVID',  color='steelblue', alpha=0.85)
ax.bar(x + w/2, tail_df['CVaR Post'], w, label='Post-COVID', color='crimson',   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(tail_df.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('CVaR 95% (daily %, negative = loss)')
ax.set_title('Conditional VaR (Expected Shortfall) — Pre vs Post COVID-19',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('tail_risk.png', bbox_inches='tight', dpi=120)
plt.show()


## Step 8 — PCA: Factor Structure of Risk

PCA reveals *how many independent risk factors* drive the cross-sectional variation of returns.
A rise in PC1 variance explained signals **higher factor concentration** — diversification breaks down.

We compare the factor structure **before** and **after** COVID.

In [ ]:
N_COMP = min(6, len(returns.columns))

def run_pca(df_period, n):
    scaled = StandardScaler().fit_transform(df_period)
    pca = PCA(n_components=n)
    pca.fit(scaled)
    return pca

pca_pre  = run_pca(pre,  N_COMP)
pca_post = run_pca(post, N_COMP)

ve_pre  = pca_pre.explained_variance_ratio_  * 100
ve_post = pca_post.explained_variance_ratio_ * 100

print('Variance Explained by Component (%)')
print(f'{"Component":<15} {"Pre-COVID":>12} {"Post-COVID":>12} {"Δ":>10}')
print('-' * 52)
for i in range(N_COMP):
    print(f'PC{i+1:<13} {ve_pre[i]:>12.2f} {ve_post[i]:>12.2f} {ve_post[i]-ve_pre[i]:>+10.2f}')
cum_pre  = np.cumsum(ve_pre)
cum_post = np.cumsum(ve_post)
print(f'\nCumulative ({N_COMP} PCs): Pre={cum_pre[-1]:.1f}%  Post={cum_post[-1]:.1f}%')

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(1, N_COMP + 1)
w = 0.35

# Scree
axes[0].bar(x - w/2, ve_pre,  w, label='Pre-COVID',  color='steelblue', alpha=0.85)
axes[0].bar(x + w/2, ve_post, w, label='Post-COVID', color='crimson',   alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'PC{i}' for i in x])
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot')
axes[0].legend()

# Cumulative
axes[1].plot(x, cum_pre,  'o-', color='steelblue', label='Pre-COVID')
axes[1].plot(x, cum_post, 's-', color='crimson',   label='Post-COVID')
axes[1].axhline(80, color='gray', linestyle=':', lw=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'PC{i}' for i in x])
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].set_title('Cumulative Scree')
axes[1].legend()

# PC1 loadings
assets_short = [c[:10] for c in returns.columns]
idx = np.arange(len(returns.columns))
axes[2].barh(idx - 0.2, pca_pre.components_[0],  0.4,
             label='Pre-COVID',  color='steelblue', alpha=0.85)
axes[2].barh(idx + 0.2, pca_post.components_[0], 0.4,
             label='Post-COVID', color='crimson',   alpha=0.85)
axes[2].set_yticks(idx)
axes[2].set_yticklabels(assets_short, fontsize=7)
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_title('PC1 Loadings')
axes[2].legend(fontsize=7)

plt.suptitle('PCA — Factor Structure of Risk: Pre vs Post COVID-19',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_analysis.png', bbox_inches='tight', dpi=120)
plt.show()


## Step 9 — Rolling Correlations: Cross-Asset Co-movement

We compute **252-day rolling correlations** for key asset pairs.
Shifts in the sign or magnitude post-COVID reveal changes in diversification properties.

In [ ]:
WINDOW = 252

pairs = [
    ('S&P500',     'US HY Bonds',          'Equities vs Credit'),
    ('S&P500',     'Gold',                  'Equities vs Gold'),
    ('S&P500',     'US T 10-year Yield',    'Equities vs Rates'),
    ('S&P500',     'Oil futures',           'Equities vs Oil'),
    ('US HY Bonds','Gold',                  'Credit vs Gold'),
    ('EURUSD',     'US T 10-year Yield',    'EUR/USD vs US Rates'),
]

fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
axes = axes.flatten()

for i, (a1, a2, label) in enumerate(pairs):
    rc = returns[[a1, a2]].rolling(WINDOW).corr().unstack()[a1][a2]
    pre_avg  = rc[rc.index <  COVID_BREAK].mean()
    post_avg = rc[rc.index >= COVID_BREAK].mean()
    axes[i].plot(rc.index, rc, color='steelblue', lw=0.7)
    axes[i].axvline(COVID_BREAK, color='red', linestyle='--', lw=1.5)
    axes[i].axhline(pre_avg,  color='steelblue', linestyle=':', lw=1.5,
                   label=f'Pre avg:  {pre_avg:.2f}')
    axes[i].axhline(post_avg, color='crimson',   linestyle=':', lw=1.5,
                   label=f'Post avg: {post_avg:.2f}')
    axes[i].axhline(0, color='black', lw=0.5, linestyle=':')
    axes[i].fill_between(rc.index, rc, alpha=0.12, color='steelblue')
    axes[i].set_ylim(-1, 1)
    axes[i].set_ylabel('Correlation', fontsize=8)
    axes[i].set_title(label, fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=7)

plt.suptitle(f'Rolling {WINDOW}-day Correlations (red = COVID break)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('rolling_correlations.png', bbox_inches='tight', dpi=120)
plt.show()

# ── Correlation heatmaps ──────────────────────────────────────
corr_pre  = pre.corr()
corr_post = post.corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, mat, title in zip(axes,
                          [corr_pre, corr_post],
                          ['Pre-COVID Correlation Matrix',
                           'Post-COVID Correlation Matrix']):
    im = ax.imshow(mat, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(mat.columns)))
    ax.set_yticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(mat.columns, fontsize=7)
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for r in range(len(mat)):
        for c in range(len(mat.columns)):
            ax.text(c, r, f'{mat.iloc[r, c]:.2f}',
                    ha='center', va='center', fontsize=5)

plt.suptitle('Correlation Matrices: Pre vs Post COVID-19',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmaps.png', bbox_inches='tight', dpi=120)
plt.show()


## Step 10 — DCC-GARCH(1,1): Dynamic Conditional Correlation

We implement **DCC-GARCH** in two stages:
1. **Stage 1** — fit univariate GARCH(1,1) to each series, extract standardised residuals ε̂_t
2. **Stage 2** — estimate DCC(1,1) on {ε̂_t}: Q_t = (1−a−b)Q̄ + a·ε_{t-1}ε_{t-1}ᵀ + b·Q_{t-1}

Parameters *a* (news impact) and *b* (persistence) are estimated by quasi-MLE.

In [ ]:
# ── Stage 1: Univariate GARCH standardised residuals ─────────
def garch_std_resid(series):
    res = arch_model(series, vol='Garch', p=1, q=1,
                     dist='normal', rescale=False).fit(disp='off')
    return (res.resid / res.conditional_volatility).values

# ── Stage 2: DCC(1,1) ─────────────────────────────────────────
def fit_dcc(E):                           # E: T×N array of std residuals
    T, N = E.shape
    Q_bar = np.cov(E.T)

    def neg_loglik(params):
        a, b = params
        if a <= 0 or b <= 0 or a + b >= 1:
            return 1e10
        Q = Q_bar.copy()
        ll = 0.0
        for t in range(T):
            e = E[t]
            Q = (1 - a - b) * Q_bar + a * np.outer(e, e) + b * Q
            d = np.sqrt(np.maximum(np.diag(Q), 1e-8))
            R = Q / np.outer(d, d)
            np.fill_diagonal(R, 1.0)
            sign, logdet = np.linalg.slogdet(R)
            if sign <= 0:
                return 1e10
            ll += logdet + float(e @ np.linalg.solve(R, e) - e @ e)
        return ll

    res = minimize(neg_loglik, [0.05, 0.90], method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-6})
    a, b = np.abs(res.x)

    Q = Q_bar.copy()
    corrs = np.zeros((T, N, N))
    for t in range(T):
        e = E[t]
        Q = (1 - a - b) * Q_bar + a * np.outer(e, e) + b * Q
        d = np.sqrt(np.maximum(np.diag(Q), 1e-8))
        R = Q / np.outer(d, d)
        np.fill_diagonal(R, 1.0)
        corrs[t] = R
    return corrs, a, b

# ── Run DCC on three meaningful pairs ────────────────────────
dcc_pairs = [
    ('S&P500',     'US HY Bonds',       'Equities vs Credit'),
    ('S&P500',     'Gold',              'Equities vs Gold'),
    ('S&P500',     'US T 10-year Yield','Equities vs Rates'),
]

fig, axes = plt.subplots(len(dcc_pairs), 1,
                         figsize=(14, 4 * len(dcc_pairs)), sharex=True)

for i, (a1, a2, label) in enumerate(dcc_pairs):
    print(f'Fitting DCC on {a1} vs {a2} ...')
    E = np.column_stack([garch_std_resid(returns[a1]),
                         garch_std_resid(returns[a2])])
    corrs, a_hat, b_hat = fit_dcc(E)
    dcc_series = pd.Series(corrs[:, 0, 1], index=returns.index)

    pre_avg  = dcc_series[dcc_series.index <  COVID_BREAK].mean()
    post_avg = dcc_series[dcc_series.index >= COVID_BREAK].mean()

    print(f'  a={a_hat:.4f}  b={b_hat:.4f}  '
          f'persistence={a_hat+b_hat:.4f}  '
          f'pre_avg={pre_avg:.3f}  post_avg={post_avg:.3f}')

    ax = axes[i]
    ax.plot(dcc_series.index, dcc_series,
            color='darkblue', lw=0.7, label='DCC correlation')
    ax.axvline(COVID_BREAK, color='red', linestyle='--', lw=1.5)
    ax.axhline(pre_avg,  color='steelblue', linestyle=':',
               lw=1.5, label=f'Pre avg: {pre_avg:.3f}')
    ax.axhline(post_avg, color='crimson',   linestyle=':',
               lw=1.5, label=f'Post avg: {post_avg:.3f}')
    ax.fill_between(dcc_series.index, dcc_series,
                    alpha=0.12, color='steelblue')
    ax.set_ylim(-1, 1)
    ax.set_ylabel('DCC ρ', fontsize=8)
    ax.set_title(f'DCC-GARCH(1,1) — {label}  '
                 f'[a={a_hat:.3f}, b={b_hat:.3f}, '
                 f'persistence={a_hat+b_hat:.3f}]',
                 fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Dynamic Conditional Correlations — DCC-GARCH(1,1)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('dcc_correlation.png', bbox_inches='tight', dpi=120)
plt.show()


## Step 10b — Risk Regimes: Structural Breaks & Markov-Switching

The course (Ielpo, Class 12) distinguishes two types of parameter instability:

| | Structural break | Markov-switching |
|---|---|---|
| **Nature** | Permanent shift | Recurring latent state |
| **Question answered** | Did risk change for good after COVID? | Was COVID a new regime or a revisit? |
| **Tools** | Chow, Quandt-Andrews, Bai-Perron | Hamilton (1989) filter |

We apply these to two signals that summarise risk structure across our full universe:
- **Rolling 252-day volatility** of S&P500 — risk level signal
- **Average pairwise correlation** across all 14 assets — risk structure signal


In [ ]:
import ruptures as rpt

ROLL_W = 252

# Signal 1 — Rolling volatility: did risk LEVEL change permanently?
roll_vol = returns["S&P500"].rolling(ROLL_W).std() * np.sqrt(252)
roll_vol = roll_vol.dropna()

# Signal 2 — Average pairwise correlation: did risk STRUCTURE change?
# This summarises co-movement across the entire 14-asset universe
pairs_corr = {}
for i, c1 in enumerate(returns.columns):
    for c2 in returns.columns[i+1:]:
        rc = (returns[[c1, c2]].rolling(ROLL_W).corr()
              .unstack()[c1][c2])
        pairs_corr[f"{c1}|{c2}"] = rc
avg_corr = pd.DataFrame(pairs_corr).mean(axis=1).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, signal, title, color in [
    (axes[0], roll_vol, "S&P500 Rolling 252-day Volatility (annualised %)", "steelblue"),
    (axes[1], avg_corr, "Average Pairwise Correlation — All 14 Assets",     "steelblue"),
]:
    ax.plot(signal.index, signal, color=color, lw=0.8)
    ax.axvline(COVID_BREAK, color="red", linestyle="--", lw=1.5, label="COVID break")
    ax.fill_between(signal.index, signal,
                    where=(signal.index >= COVID_BREAK),
                    alpha=0.12, color="red")
    pre_m  = signal[signal.index <  COVID_BREAK].mean()
    post_m = signal[signal.index >= COVID_BREAK].mean()
    ax.axhline(pre_m,  color="steelblue", linestyle=":", lw=1.3,
               label=f"Pre avg: {pre_m:.3f}")
    ax.axhline(post_m, color="crimson",   linestyle=":", lw=1.3,
               label=f"Post avg: {post_m:.3f}")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Risk-Structure Signals for Break & Regime Analysis",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("regime_signals.png", bbox_inches="tight", dpi=120)
plt.show()


### A. Chow Test — Known Break at COVID (2020-03-11)

$$F_{Chow} = \frac{(SSR_0 - SSR_1 - SSR_2)\,/\,k}{(SSR_1 + SSR_2)\,/\,(T-2k)} \sim F(k,\,T-2k)$$

$H_0$: the mean of the signal is **identical** before and after COVID.  
Rejection = evidence of a **permanent structural shift**.


In [ ]:
def chow_test(series, break_date, k=1):
    """Chow (1960) — OLS SSR formulation, constant-only regression."""
    y  = series.values
    y1 = series[series.index <  break_date].values
    y2 = series[series.index >= break_date].values
    T  = len(y)
    SSR0 = np.sum((y  - y.mean())  ** 2)
    SSR1 = np.sum((y1 - y1.mean()) ** 2)
    SSR2 = np.sum((y2 - y2.mean()) ** 2)
    F    = ((SSR0 - SSR1 - SSR2) / k) / ((SSR1 + SSR2) / (T - 2*k))
    p    = 1 - stats.f.cdf(F, dfn=k, dfd=T - 2*k)
    return F, p

# Apply to both signals AND to individual asset returns
test_signals = {
    "S&P500 rolling volatility":            roll_vol,
    "Avg pairwise correlation (14 assets)": avg_corr,
}
for col in ["S&P500", "Eurostoxx 50", "US HY Bonds", "Gold",
            "US T 10-year Yield", "Oil futures"]:
    test_signals[f"{col} returns"] = returns[col]

print("CHOW TEST — H0: no structural break at COVID (2020-03-11)")
print(f"{'Signal':<42} {'F-stat':>9} {'p-value':>9} {'Reject H0?':>12}")
print("-" * 75)
for label, sig in test_signals.items():
    F, p = chow_test(sig, COVID_BREAK)
    flag = "YES (1%)" if p < 0.01 else "YES (5%)" if p < 0.05 else "no"
    print(f"{label:<42} {F:>9.4f} {p:>9.4f} {flag:>12}")


### B. Quandt-Andrews — Unknown Break Date

When the break date is unknown, we sweep all admissible candidates (15% trim)
and take the supremum (Andrews, 1993):

$$\sup F = \sup_{\tau \in \Lambda_T}\, F_T(\tau)$$

This answers: *if a break exists, when did it occur — and does it align with COVID?*  
Andrews (1993) 5% critical value: **8.85** for $k=1$.


In [ ]:
def quandt_andrews(series, trim=0.15):
    n = len(series)
    t0, t1 = int(n * trim), int(n * (1 - trim))
    F_path, sup_F, sup_date = {}, 0.0, None
    for t in range(t0, t1):
        F_t, _ = chow_test(series, series.index[t])
        F_path[series.index[t]] = F_t
        if F_t > sup_F:
            sup_F, sup_date = F_t, series.index[t]
    return pd.Series(F_path), sup_F, sup_date

CV_5PCT = 8.85

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, (label, sig) in zip(axes, [
    ("S&P500 Rolling Volatility",            roll_vol),
    ("Avg Pairwise Correlation (14 assets)", avg_corr),
]):
    F_path, sup_F, sup_date = quandt_andrews(sig)
    ax.plot(F_path.index, F_path, color="steelblue", lw=0.8, label="Chow F(τ)")
    ax.axhline(CV_5PCT, color="red", linestyle="--", lw=1.5,
               label=f"5% CV = {CV_5PCT} (Andrews 1993)")
    ax.axvline(sup_date, color="darkorange", linestyle="--", lw=1.5,
               label=f"sup-F = {sup_F:.2f} on {sup_date.date()}")
    ax.axvline(COVID_BREAK, color="crimson", linestyle=":", lw=1.2,
               label=f"COVID (2020-03-11)")
    ax.set_ylabel("F-statistic")
    ax.set_title(f"Quandt-Andrews — {label}", fontweight="bold")
    ax.legend(fontsize=8)
    gap = abs((sup_date - COVID_BREAK).days)
    print(f"{label}:")
    print(f"  sup-F = {sup_F:.4f} | detected break: {sup_date.date()} "
          f"| gap with COVID: {gap} days "
          f"| Significant: {'YES' if sup_F > CV_5PCT else 'NO'}\n")

plt.suptitle("Quandt-Andrews: Does the Detected Break Coincide with COVID?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("quandt_andrews.png", bbox_inches="tight", dpi=120)
plt.show()


### C. Bai-Perron — Multiple Unknown Breaks

Bai & Perron (1998, 2003) minimise the total SSR globally via dynamic programming,
detecting **multiple** breaks without imposing any date a priori.

Applied to the **average pairwise correlation** — our broadest risk-structure signal —
to identify distinct risk regimes across the full 1990–2025 sample.


In [ ]:
sig   = avg_corr
vals  = sig.values.reshape(-1, 1)
idx   = sig.index

algo     = rpt.Dynp(model="l2", min_size=120, jump=5).fit(vals)
bk_idx   = algo.predict(n_bkps=3)
bp_dates = [idx[min(b-1, len(idx)-1)] for b in bk_idx[:-1]]
bounds   = [idx[0]] + bp_dates + [idx[-1]]

segments = []
for j in range(len(bounds)-1):
    mask = (sig.index >= bounds[j]) & (sig.index <= bounds[j+1])
    sub  = sig[mask]
    segments.append({"label": f"{bounds[j].year}–{bounds[j+1].year}",
                     "start": bounds[j], "end": bounds[j+1], "mean": sub.mean()})

print("BAI-PERRON — Average Pairwise Correlation")
for bd in bp_dates:
    print(f"  Break: {bd.date()}")
print()
for seg in segments:
    print(f"  {seg['label']:<15} avg corr = {seg['mean']:>7.4f}")

shading = ["lightblue","lightcoral","lightblue","lightcoral"]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sig.index, sig, color="steelblue", lw=0.8, zorder=3,
        label="Avg pairwise corr (14 assets)")
for j, seg in enumerate(segments):
    ax.axvspan(seg["start"], seg["end"], alpha=0.25, color=shading[j])
    ax.hlines(seg["mean"], seg["start"], seg["end"],
              colors="darkblue", linewidths=2, linestyles="--",
              label=f"{seg['label']}: {seg['mean']:.3f}")
for bd in bp_dates:
    ax.axvline(bd, color="black", linestyle="--", lw=1.5)
ax.axvline(COVID_BREAK, color="red", linestyle=":", lw=1.5, label="COVID break")
ax.axhline(0, color="black", lw=0.4, linestyle=":")
ax.set_ylabel("Average Pairwise Correlation")
ax.set_title("Bai-Perron Structural Breaks — Average Pairwise Correlation (14 Assets)",
             fontweight="bold")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("bai_perron.png", bbox_inches="tight", dpi=120)
plt.show()


### D. Markov-Switching — Did COVID Create a New Regime or Revisit an Old One?

A structural break says the parameter *stayed* different after COVID.
A Markov-switching model asks the deeper question: *is this a new state, or a
pre-existing high-risk state the market was already cycling through?*

$$r_t = \mu_{s_t} + \sigma_{s_t}\,\varepsilon_t, \quad s_t \in \{\text{low-vol},\, \text{high-vol}\}$$

We apply it to **4 assets representing distinct risk dimensions** — equity, credit,
safe-haven, and rates — and compare the share of time spent in the high-volatility
regime before and after COVID.


In [ ]:
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

MS_ASSETS = {
    "S&P500":             "Equity risk",
    "US HY Bonds":        "Credit risk",
    "Gold":               "Safe-haven",
    "US T 10-year Yield": "Rate risk",
}

fig, axes = plt.subplots(len(MS_ASSETS), 1,
                         figsize=(14, 4 * len(MS_ASSETS)), sharex=True)

ms_summary = []
for ax, (asset, dimension) in zip(axes, MS_ASSETS.items()):
    ms  = MarkovRegression(returns[asset], k_regimes=2,
                           trend="c", switching_variance=True).fit(disp=False)
    s2  = [ms.params["sigma2[0]"], ms.params["sigma2[1]"]]
    hr  = int(np.argmax(s2))       # high-vol regime index
    lr  = 1 - hr

    prob_h = ms.smoothed_marginal_probabilities[hr].copy()
    prob_h.index = returns.index

    pct_pre  = prob_h[prob_h.index <  COVID_BREAK].mean() * 100
    pct_post = prob_h[prob_h.index >= COVID_BREAK].mean() * 100
    dur_high = 1 / (1 - ms.regime_transition[hr, hr])

    ms_summary.append({
        "Asset":           asset,
        "Dimension":       dimension,
        "σ low-vol":       f"{np.sqrt(s2[lr]):.4f}",
        "σ high-vol":      f"{np.sqrt(s2[hr]):.4f}",
        "Dur high (days)": f"{dur_high:.0f}",
        "% high pre":      f"{pct_pre:.1f}%",
        "% high post":     f"{pct_post:.1f}%",
        "Δ":               f"{pct_post - pct_pre:+.1f}pp",
    })

    # Plot
    ax.plot(returns.index, returns[asset], color="steelblue", lw=0.4, alpha=0.6)
    ax.fill_between(returns.index,
                    returns[asset].min(), returns[asset].max(),
                    where=(prob_h > 0.5),
                    color="crimson", alpha=0.2, label="High-vol regime (P>0.5)")
    ax.axvline(COVID_BREAK, color="red", linestyle="--", lw=1.5, label="COVID break")
    ax.set_title(f"{asset} — {dimension}  "
                 f"[Pre: {pct_pre:.0f}% high-vol | Post: {pct_post:.0f}% high-vol]",
                 fontsize=9, fontweight="bold")
    ax.set_ylabel("Return (%)", fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle("Markov-Switching: Time in High-Volatility Regime — Pre vs Post COVID",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("markov_switching.png", bbox_inches="tight", dpi=120)
plt.show()

print("\nMARKOV-SWITCHING SUMMARY TABLE")
print(pd.DataFrame(ms_summary).set_index("Asset").to_string())


## Step 10c — Risk Transmission: Granger Causality

The instructions explicitly list risk transmission as a dimension to investigate.
**Granger causality** tests whether past returns of asset A help predict future
returns of asset B, beyond what B's own past already explains.

We compare Granger relationships **pre vs post COVID** to assess whether the
transmission of risk across assets has changed — i.e. whether shocks propagate
differently in the post-COVID environment.

We test 5 economically motivated directed pairs:
- S&P500 → US HY Bonds (does equity stress transmit to credit?)
- S&P500 → Gold (does equity stress drive safe-haven demand?)
- S&P500 → Oil futures (equity-commodity linkage)
- US HY Bonds → US T 10-year Yield (credit stress → rate market)
- Eurostoxx 50 → S&P500 (cross-geography spillover)


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

GRANGER_PAIRS = [
    ("S&P500",       "US HY Bonds",          "Equity → Credit"),
    ("S&P500",       "Gold",                  "Equity → Safe-haven"),
    ("S&P500",       "Oil futures",           "Equity → Commodity"),
    ("US HY Bonds",  "US T 10-year Yield",    "Credit → Rates"),
    ("Eurostoxx 50", "S&P500",                "EU Equity → US Equity"),
]

MAX_LAG = 5   # test up to 5 lags, report minimum p-value

def granger_pval(cause, effect, data, maxlag=MAX_LAG):
    """Return minimum p-value across lags (F-test)."""
    df_test = data[[effect, cause]].dropna()
    results = grangercausalitytests(df_test, maxlag=maxlag, verbose=False)
    pvals   = [results[lag][0]["ssr_ftest"][1] for lag in range(1, maxlag+1)]
    return min(pvals)

rows = []
for cause, effect, label in GRANGER_PAIRS:
    p_pre  = granger_pval(cause, effect, pre)
    p_post = granger_pval(cause, effect, post)
    rows.append({
        "Pair":            label,
        "p-value Pre":     round(p_pre,  4),
        "Sig Pre":         "YES" if p_pre  < 0.05 else "no",
        "p-value Post":    round(p_post, 4),
        "Sig Post":        "YES" if p_post < 0.05 else "no",
        "Change":          "appeared" if p_pre >= 0.05 and p_post < 0.05
                           else "disappeared" if p_pre < 0.05 and p_post >= 0.05
                           else "stable"
    })

granger_df = pd.DataFrame(rows).set_index("Pair")
print("GRANGER CAUSALITY — Pre vs Post COVID (H0: no causality, max lag = 5)")
print("Rejection at 5% = statistically significant transmission")
print()
print(granger_df.to_string())

# ── Visual: p-value comparison ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
x  = np.arange(len(granger_df))
w  = 0.35
ax.bar(x - w/2, granger_df["p-value Pre"],  w, label="Pre-COVID",
       color="steelblue", alpha=0.85)
ax.bar(x + w/2, granger_df["p-value Post"], w, label="Post-COVID",
       color="crimson",   alpha=0.85)
ax.axhline(0.05, color="black", linestyle="--", lw=1.2, label="5% threshold")
ax.set_xticks(x)
ax.set_xticklabels(granger_df.index, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Minimum p-value (across lags)")
ax.set_title("Granger Causality: Has Risk Transmission Changed Since COVID?",
             fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("granger_causality.png", bbox_inches="tight", dpi=120)
plt.show()


## Step 11 — Summary: Has the Structure of Risk Changed?

In [ ]:
print('='*72)
print('SUMMARY — STRUCTURE OF RISK: PRE vs POST COVID-19')
print('='*72)

# Annualised vol change for S&P500
sp_vol_pre  = pre['S&P500'].std()  * np.sqrt(252)
sp_vol_post = post['S&P500'].std() * np.sqrt(252)

# CVaR for S&P500
cvar_pre  = historical_cvar(pre['S&P500'])
cvar_post = historical_cvar(post['S&P500'])

# Average pairwise correlation
avg_corr_pre  = pre.corr().values[np.triu_indices_from(pre.corr().values, k=1)].mean()
avg_corr_post = post.corr().values[np.triu_indices_from(post.corr().values, k=1)].mean()

rows = [
    ('S&P500 Ann. Volatility (%)',
     f'{sp_vol_pre:.2f}',  f'{sp_vol_post:.2f}',
     f'{sp_vol_post-sp_vol_pre:+.2f}'),

    ('S&P500 CVaR 95% (daily %)',
     f'{cvar_pre:.3f}',   f'{cvar_post:.3f}',
     f'{cvar_post-cvar_pre:+.3f}'),

    ('PC1 Variance Explained (%)',
     f'{ve_pre[0]:.1f}',  f'{ve_post[0]:.1f}',
     f'{ve_post[0]-ve_pre[0]:+.1f}'),

    ('Avg Pairwise Correlation',
     f'{avg_corr_pre:.3f}', f'{avg_corr_post:.3f}',
     f'{avg_corr_post-avg_corr_pre:+.3f}'),
]

print(f'\n{"Dimension":<35} {"Pre-COVID":>12} {"Post-COVID":>12} {"Change":>10}')
print('-'*72)
for row in rows:
    print(f'{row[0]:<35} {row[1]:>12} {row[2]:>12} {row[3]:>10}')

print("""
─────────────────────────────────────────────────────────────────────────
CONCLUSION
─────────────────────────────────────────────────────────────────────────
The evidence points to a PARTIAL structural change in the risk landscape:

  ✓ RISK LEVELS rose across nearly all assets post-COVID, with higher
    GARCH-estimated conditional volatility and worse tail risk metrics.

  ✓ FACTOR CONCENTRATION increased: PC1 explains more variance post-COVID,
    implying a more correlated, harder-to-diversify universe.

  ✓ CROSS-ASSET CORRELATIONS shifted, particularly for the equity-bond
    relationship — a key pillar of traditional 60/40 portfolio construction.

  ~ Some correlation patterns NORMALISED after the acute 2020 phase,
    suggesting the shift is partial rather than permanent.
─────────────────────────────────────────────────────────────────────────
""")
